## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

I0000 00:00:1774189871.832047  448971 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774189871.869159  448971 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1774189872.730663  448971 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        # 声明模型参数
        # 使用 He 初始化 (stddev = sqrt(2/n_in))，适配 ReLU 激活函数
        # 第一层：输入784维（28*28），输出256维
        self.W1 = tf.Variable(tf.random.normal([784, 256], stddev=np.sqrt(2.0/784)))
        self.b1 = tf.Variable(tf.zeros([256]))
        # 第二层：输入256维，输出10维（10个类别）
        self.W2 = tf.Variable(tf.random.normal([256, 10], stddev=np.sqrt(2.0/256)))
        self.b2 = tf.Variable(tf.zeros([10]))

    def __call__(self, x):
        # 将输入展平为 (batch_size, 784)
        x = tf.reshape(x, [-1, 784])
        # 第一层全连接 + ReLU 激活
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        # 第二层全连接，输出 logits（未归一化）
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

W0000 00:00:1774189875.269881  448971 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.525857 ; accuracy 0.09431667


epoch 1 : loss 2.5047958 ; accuracy 0.09755


epoch 2 : loss 2.4846282 ; accuracy 0.10056667


epoch 3 : loss 2.4652786 ; accuracy 0.1042


epoch 4 : loss 2.446673 ; accuracy 0.1081


...

epoch 45 : loss 1.9639943 ; accuracy 0.42791668


epoch 46 : loss 1.9552488 ; accuracy 0.43581668


epoch 47 : loss 1.9465727 ; accuracy 0.44346666


epoch 48 : loss 1.937963 ; accuracy 0.45135


epoch 49 : loss 1.9294183 ; accuracy 0.45865
test loss 1.9134643 ; accuracy 0.4693
